In [ ]:
language = "pt"

In [4]:
# 1. Gravação de áudio com Python (e uma pitada de JavaScript)
# Referência: https://gist.github.com/korakot/c21c3476c024ad6d56d5f48b0bca92be

from IPython.display import Audio, display, Javascript
from google.collab import output 
from base64 import b64decode

# Código JavaScript para gravar áudio do usuário usando a "MediaStream Recording API"
RECORD = """
const sleep  = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async ()=>{
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

def record(sec=5):
    display(Javascript(RECORD))
    js_result = output.eval_js('record(%s)' % (sec * 1000))
    audio = b64decode(js_result.split(',')[1])
    file_name = 'request_audio.wav'
    with open(file_name, 'wb') as f:
      f.write(audio)
    return f'/content/{file_name}'  

record_file = record()
display(audio(record_file, autoplay=True))

ModuleNotFoundError: No module named 'google'

In [ ]:
# 2. Reconhecimento de Fala com Whisper (OpenAI)
pip install git+https://github.com/openai/whisper.git -q

import whisper

model = whisper.load_model("small")

result = model.transcribe(record_file, fp16=False, language=language)
transcription = result['text']
print(transcription)

In [ ]:
# 3. Integração com a API do ChatGPT
pip install openai

import openai

client = openai.OpenAI(api_key='')

response = client.chat.completions.create(
    model='gpt-3.5-turbo',
    messages=[ { 'role': 'user', 'content': transcription } ]
)

chatgpt_response = response.choices[0].message.content
print(chatgpt_response)

In [ ]:
# 4. Sintetizando a Resposta do ChatGPT como Voz (gTTS)
pip install gTTS 

from gtts import gTTS 

gtts_object = gTTS(text=chatgpt_response, lang=language, slow=False)
response_audio = '/content/response_audio.wav'
gtts_object.save(response_audio)
display(Audio(response_audio, autoplay=True))